In [8]:
from google.colab import drive
import pandas as pd
import numpy as np
import os
import shutil
import matplotlib.pyplot as plt

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
all_data = pd.read_csv('/content/drive/MyDrive/pioneer_data/all_data.csv')

In [4]:
all_data.head()

,dir_num,convo_id,user_id,channel,enjoyment,enjoy_diff
0,1,67836c1d-1334-41a0-a33a-4f788e8b6fb3,5cc604bd3bbb120018a3c0e2,L,7.0,0.0
1,1,67836c1d-1334-41a0-a33a-4f788e8b6fb3,5e0acbffdc79bd35336ed6e0,R,7.0,0.0
2,1,8a852635-85e0-4b09-8004-ad3d7de925d5,5ef6bb87b83f3b000a3f04ae,L,9.0,2.0
3,1,8a852635-85e0-4b09-8004-ad3d7de925d5,55a43cf3fdf99b02ff6cb0b4,R,7.0,2.0
4,1,c97f8a32-6249-409c-b042-aaed261c3f64,5f481f85c6a7d02c489665f2,L,8.0,0.0


In [45]:
asym_data = all_data[abs(all_data['enjoy_diff']) >= 4]
# remove faulty convo
asym_data = asym_data[asym_data['convo_id'] != '7fc59ebb-a980-4df1-9d39-f15ae25f3f27']
print(len(asym_data))
asym_data.head()


244


,dir_num,convo_id,user_id,channel,enjoyment,enjoy_diff
40,3,46f8e9b8-f80a-48cf-90a0-2e29908202c0,5d5eeb06d8bcde00162d73f1,L,2.0,-4.0
41,3,46f8e9b8-f80a-48cf-90a0-2e29908202c0,57656f6c2bfddf000125cce5,R,6.0,-4.0
58,4,a4fa5355-74ba-4622-a58b-57335efc8a9e,56802e5fc5767f00121cc6a0,L,9.0,4.0
59,4,a4fa5355-74ba-4622-a58b-57335efc8a9e,5ecf4b18f7b0443609e07646,R,5.0,4.0
126,7,5a307dec-0265-4dab-ade6-bc6392695c9e,5e7072e5fb136c63e94f3fa4,L,5.0,-4.0


In [5]:
if os.path.exists('/content/drive/MyDrive/pioneer_data/pseudo_metadata.csv'):
  # if pseudopair metadata already exists, read from it + skip df building

  print(f'metadata found...')
  print(f'reading in pseudo metadata file as df')
  pseudo_data = pd.read_csv('/content/drive/MyDrive/pioneer_data/pseudo_metadata.csv')
else:
  # construct pseudopairing metadata

  print(f'metadata not found...')
  print(f'building pseudopairing metadata...')

  rows = [] # convert to df later
  used = set() # ensure non-repetition

  np.random.seed(42)

  size = 100
  count = 0

  # make two sets of data based on CANDOR's assigned channels
  left = asym_data[asym_data['channel'] == 'L'].reset_index(drop=True)
  right = asym_data[asym_data['channel'] == 'R'].reset_index(drop=True)

  while count < size:
    u1 = left.sample(1).iloc[0]
    u2 = right.sample(1).iloc[0]

    # same convo: exclude
    if u1['convo_id'] == u2['convo_id']:
      print(f'same conversation found; excluding...')
      continue

    # same person: exclude
    if u1['user_id'] == u2['user_id']:
      print(f'same person found; excluding...')
      continue

    # same pseudopair already exists
    check = (u1['user_id'], u2['user_id'])
    if check in used:
      print(f'existing pseudopairing exists; excluding...')
      continue

    used.add(check)

    rows.append({'pair_id': count, **u1.to_dict()})
    rows.append({'pair_id': count, **u2.to_dict()})

    count += 1

    pseudo_data = pd.DataFrame(rows)

pseudo_data.to_csv('/content/drive/MyDrive/pioneer_data/pseudo_metadata.csv', index=False)
print(f'{len(pseudo_data)} rows total, {pseudo_data['pair_id'].nunique()} pairs')
pseudo_data.head()

metadata found...
reading in pseudo metadata file as df
200 rows total, 100 pairs


,pair_id,dir_num,convo_id,user_id,channel,enjoyment,enjoy_diff
0,0,36,b88c01b7-4d38-42c5-b650-ee796b3ef869,5dd33bd8f01cbe3350807cf6,L,4.0,-4.0
1,0,12,2fb5e5af-52fe-4469-b77a-0b4467ca0bf1,5d54d3560c5b0400174b3c81,R,4.0,4.0
2,1,12,2fb5e5af-52fe-4469-b77a-0b4467ca0bf1,5e2be6efe179572183644844,L,8.0,4.0
3,1,49,671f9e6e-a35f-4bdc-9b61-4adb53bb1690,5e531b2205acdb33c0f5f24c,R,8.0,-5.0
4,2,106,f00d9688-27fe-45ce-9d26-e38f729e8641,5d0fb996286e1700010de2c1,L,4.0,-4.0


In [7]:
for pid in pseudo_data['pair_id'].unique():
  pair = pseudo_data['pair_id'] == pid
  left = pseudo_data.loc[pair & (pseudo_data['channel'] == 'L'), 'enjoyment'].item()
  right = pseudo_data.loc[pair & (pseudo_data['channel'] == 'R'), 'enjoyment'].item()
  pseudo_data.loc[pair, 'enjoy_diff'] = left - right

pseudo_data.head(10)


,pair_id,dir_num,convo_id,user_id,channel,enjoyment,enjoy_diff
0,0,36,b88c01b7-4d38-42c5-b650-ee796b3ef869,5dd33bd8f01cbe3350807cf6,L,4.0,0.0
1,0,12,2fb5e5af-52fe-4469-b77a-0b4467ca0bf1,5d54d3560c5b0400174b3c81,R,4.0,0.0
2,1,12,2fb5e5af-52fe-4469-b77a-0b4467ca0bf1,5e2be6efe179572183644844,L,8.0,0.0
3,1,49,671f9e6e-a35f-4bdc-9b61-4adb53bb1690,5e531b2205acdb33c0f5f24c,R,8.0,0.0
4,2,106,f00d9688-27fe-45ce-9d26-e38f729e8641,5d0fb996286e1700010de2c1,L,4.0,-1.0
5,2,143,94ac8821-51cc-4d9c-a185-b98cbdcd938b,5e4aedd76f4883000e48bfe4,R,5.0,-1.0
6,3,54,36f338b1-bb0e-4c79-b40d-1c584913f262,5f0963c6310eaf000b922e94,L,5.0,2.0
7,3,134,d5d67a5a-4b95-479f-a4a8-cb86ae7aea54,5db6f5c8c5ffeb000ac7cfb2,R,3.0,2.0
8,4,48,8f1d8967-f1de-4a3f-a5fc-3adf9015e732,5e5ca080bb37ff47b09dd4cc,L,5.0,3.0
9,4,135,16af37de-347d-4826-8a60-691d9a0e7fea,5d49a0b751695e0001a188a8,R,2.0,3.0


In [10]:
pseudo_data.to_csv('/content/drive/MyDrive/pioneer_data/pseudo_metadata.csv', index=False)

In [49]:
# make pseudopairing .csv au files

BASE = '/content/drive/MyDrive/au_activity'
REAL = f'{BASE}/real'
PSEUDO = f'{BASE}/pseudopairs'
MIN_FRAMES = 30

asym_data = pd.read_csv('/content/drive/MyDrive/pioneer_data/asym_data.csv')
pseudo_data = pd.read_csv('/content/drive/MyDrive/pioneer_data/pseudo_metadata.csv')

skipped = []
count = 0

# iterate by pair_id
for pid, grp in pseudo_data.groupby('pair_id'):
  print(f'building pair {pid+1}/{pseudo_data['pair_id'].nunique()}')

  left = grp[grp['channel'] == 'L']
  right = grp[grp['channel'] == 'R']

  if len(left) != 1 or len(right) != 1:
    skipped.append(pid)
    print(f'skipped {pid}: len(left) = {len(left)}, len(right) = {len(right)}')
    continue

  # get convo ids for both speakers
  l_cid = left.iloc[0]['convo_id']
  r_cid = right.iloc[0]['convo_id']

  # use preprocessed streams
  l_path = f'{REAL}/{l_cid}/p_left.csv'
  r_path = f'{REAL}/{r_cid}/p_right.csv'

  if not os.path.exists(l_path) or not os.path.exists(r_path):
    skipped.append(pid)
    print(f'skipped {pid}: {l_path} or {r_path} does not exist')
    continue

  l_df = pd.read_csv(l_path)
  r_df = pd.read_csv(r_path)

  n = min(len(l_df), len(r_df))
  if n < MIN_FRAMES:
    skipped.append(pid)
    print(f'skipped {pid}: min length ({n}) < {MIN_FRAMES}')
    continue

  # crop to the shared end of both videos
  l_df = l_df.iloc[:n].reset_index(drop=True)
  r_df = r_df.iloc[:n].reset_index(drop=True)

  # og video frames differ -> overwrite
  frames = np.arange(n) * 5
  l_df['frame'] = frames
  r_df['frame'] = frames

  # build directory based off pair id
  out_dir = f'{PSEUDO}/{pid}'
  os.makedirs(out_dir, exist_ok=True)

  # copy unprocessed video streams to dir
  shutil.copy(f'{REAL}/{l_cid}/left.csv', f'{out_dir}/left.csv')
  shutil.copy(f'{REAL}/{r_cid}/right.csv', f'{out_dir}/right.csv')

  # save processed video streams to dir
  l_df.to_csv(f'{out_dir}/p_left.csv', index=False)
  r_df.to_csv(f'{out_dir}/p_right.csv', index=False)

  count += 1

print(f'build {count}/{pseudo_data['pair_id'].nunique()} pseudopairs')
print(f'skipped {len(skipped)} pairs')

building pair 1/100
building pair 2/100
building pair 3/100
building pair 4/100
building pair 5/100
building pair 6/100
building pair 7/100
building pair 8/100
building pair 9/100
building pair 10/100
building pair 11/100
building pair 12/100
building pair 13/100
building pair 14/100
building pair 15/100
building pair 16/100
building pair 17/100
building pair 18/100
building pair 19/100
building pair 20/100
building pair 21/100
building pair 22/100
building pair 23/100
building pair 24/100
building pair 25/100
building pair 26/100
building pair 27/100
building pair 28/100
building pair 29/100
building pair 30/100
building pair 31/100
building pair 32/100
building pair 33/100
building pair 34/100
building pair 35/100
building pair 36/100
building pair 37/100
building pair 38/100
building pair 39/100
building pair 40/100
building pair 41/100
building pair 42/100
building pair 43/100
building pair 44/100
building pair 45/100
building pair 46/100
building pair 47/100
building pair 48/100
b